### Environment Setup

In [1]:
pip install elevenlabs pydub python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install resemblyzer librosa soundfile scipy pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import json
import os
import subprocess
from dotenv import load_dotenv
from elevenlabs.client import ElevenLabs

### Config

In [ ]:
INPUT_JSON = "2_dialogues/scenario3_d1.json"
TEMP_DIR = "temp_audio/scenario3_a1"
os.makedirs(TEMP_DIR, exist_ok=True)
OUTPUT_AUDIO = os.path.join("3_generated_audios", "scenario3_audio1.wav")

PAUSE_SAME_SPEAKER = 0.3
PAUSE_DIFFERENT_SPEAKER = 1.2
PAUSE_TOPIC_CHANGE = 2.5

load_dotenv()
api_key = os.getenv("ELEVENLABS_API_KEY")
client = ElevenLabs(api_key=api_key)
FFMPEG_PATH = r"C:\Users\mleon\Downloads\ffmpeg-8.1-essentials_build\ffmpeg-8.1-essentials_build\bin\ffmpeg.exe"

SPEAKER_VOICES = {
    "Command": "nPczCjzI2devNBz1zQrb",   # Brian
    "Engine 1": "bIHbv24MWmeRgasZH58o",  # Will
    "Rescue 2": "Xb7hH8MSUJpSbSDYk0k2",  # Alice
    "Alpha": "cjVigY5qzO86Huf0OWal",     # Eric
    "Beta": "IKne3meq5aSn9XLyUdCD",      # Charlie
    "Unit Phi": "EXAVITQu4vr4xnSDxMaL",  # Sarah
}

### Audio Generation

In [ ]:
def generate_tts(text, voice, filename):
    audio = client.text_to_speech.convert(
        text=text,
        voice_id=voice,
        model_id="eleven_multilingual_v2",
        voice_settings={
            "stability": 0.25,
            "similarity_boost": 0.7,
            "style": 0.0,
            "use_speaker_boost": True,
        },
    )

    with open(filename, "wb") as f:
        for chunk in audio:
            f.write(chunk)

def create_silence(filename, duration):
    subprocess.run(
        [
            FFMPEG_PATH,
            "-y",
            "-f",
            "lavfi",
            "-i",
            "anullsrc=r=44100:cl=mono",
            "-t",
            str(duration),
            filename,
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

def concat_with_pauses(dialogue, audio_files, output):
    files_with_pauses = []

    for i, file in enumerate(audio_files):
        files_with_pauses.append(file)

        if i < len(audio_files) - 1:
            current_speaker = dialogue[i]["speaker"]
            next_speaker = dialogue[i + 1]["speaker"]

            current_topic = dialogue[i].get("topic")
            next_topic = dialogue[i + 1].get("topic")

            if (
                current_topic is not None
                and next_topic is not None
                and current_topic != next_topic
            ):
                pause = PAUSE_TOPIC_CHANGE
            elif current_speaker == next_speaker:
                pause = PAUSE_SAME_SPEAKER
            else:
                pause = PAUSE_DIFFERENT_SPEAKER

            silence_file = os.path.join(TEMP_DIR, f"silence_{i}.wav")
            create_silence(silence_file, pause)
            files_with_pauses.append(silence_file)

    list_file = os.path.join(TEMP_DIR, "file_list.txt")

    with open(list_file, "w") as f:
        for file in files_with_pauses:
            f.write(f"file '{os.path.abspath(file)}'\n")

    subprocess.run(
        [
            FFMPEG_PATH,
            "-y",
            "-f",
            "concat",
            "-safe",
            "0",
            "-i",
            list_file,
            "-c",
            "copy",
            output,
        ]
    )

def apply_radio_filter(input_file, output_file):
    subprocess.run(
        [
            FFMPEG_PATH,
            "-y",
            "-i",
            input_file,
            "-af",
            "highpass=300,lowpass=3000,acompressor=threshold=-20dB:ratio=2, atempo=1.15",
            output_file,
        ]
    )

In [ ]:
# Main Pipeline
audio_files = []

with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

dialogue = data

for i, line in enumerate(dialogue):
    speaker = line["speaker"]
    message = line["message"]

    voice = SPEAKER_VOICES.get(speaker, "Rachel")
    filename = os.path.join(TEMP_DIR, f"{i}.wav")

    print(f"[{i}] {speaker}: {message}")

    generate_tts(message, voice, filename)
    audio_files.append(filename)

temp_full = os.path.join(TEMP_DIR, "full.wav")
concat_with_pauses(dialogue, audio_files, temp_full)

apply_radio_filter(temp_full, OUTPUT_AUDIO)

print(f"\nDone: {OUTPUT_AUDIO}")

[0] Command: Unit Phi here Command. Report current fire conditions and spread direction. Listening
[1] Unit Phi: Command here Unit Phi. Active wildfire in the forested area, multiple trees involved, heavy smoke moving across the clearing. Listening
[2] Command: Unit Phi here Command. Confirm any exposures or access concerns near the fire front. Listening
[3] Unit Phi: Command here Unit Phi. There appears to be a house beyond the tree line, but occupancy is not confirmed. Fire is moving in that direction through the trees. Listening
[4] Command: Unit Phi here Command. Received. Prioritize exposure assessment from a safe position and confirm if any persons are visible near that house. Listening
[5] Unit Phi: Command here Unit Phi. Negative visual confirmation of persons at this time. Smoke is limiting visibility near the house and access path. Listening

Done: 3_generated_audios\scenario3_audio1.wav
